![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Workout - Condicionales en LangGraph

Anteriormente vimos con LangGraph que el flujo era **lineal**: `normalizar → proponer` (siempre el mismo camino).

Aquí añadimos **edges condicionales**: según la intención del pedido, el grafo elige una rama.

Las ramas **emularían** el flujo de usar tres tools (preferencias / eventos / horario), pero **aún no ejecutan tools porque no han sido implementadas**. Solo cambian el prompt. Más adelante la condición típica será “¿hay tool calls?”.

Misma API key (`GEMINI_API_KEY`) y mismo wrapper `langchain-google-genai`.

## 1. Dependencias

Instala (o actualiza) las librerías necesarias. Descomenta la celda si las necesitas en tu entorno.


In [ ]:
# %pip install -qU langgraph langchain-google-genai grandalf python-dotenv

## 2. API key

Cargamos `GEMINI_API_KEY` desde `.env` o, si falta, la pedimos por consola con `getpass` (no se imprime en claro).

In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

print("GEMINI_API_KEY:", "sí" if os.getenv("GEMINI_API_KEY") else "no")


GEMINI_API_KEY: sí


## 3. Construcción del flujo con bifurcación. Edges condicionales

Montamos un **flujo con bifurcación**: primero clasificamos la intención y luego el grafo elige **una** de cuatro ramas.

```text
                    ┌→ preferencias → END
                    ├→ eventos      → END
[START] → clasificar ┤
                    ├→ horario      → END
                    └→ general      → END
```

1. **Clasificar** — Gemini etiqueta el pedido (`preferencias` / `eventos` / `horario` / `general`).
2. **Router** — una función Python lee `intencion` del estado y decide el siguiente nodo.
3. **Rama** — cada nodo responde con un prompt distinto (aún sin tools reales).

Pieza nueva respecto al notebook anterior:

| Pieza | Qué es | Aquí |
|-------|--------|------|
| **Edge fija** | Siempre A → B | `START → clasificar`, cada rama → `END` |
| **Edge condicional** | A → ? según una función | `add_conditional_edges` + `router` |

Al final compilamos y probamos varios pedidos para ver que cada uno cae en una rama distinta.

## 4. Estado + modelo

Definimos el **estado compartido** (`TypedDict`) y el modelo Gemini.

- `pedido`: texto del usuario
- `intencion`: etiqueta que escribe el nodo clasificar (la usa el router)
- `respuesta`: texto final de la rama elegida


In [2]:
from typing import TypedDict, Literal
from langchain_google_genai import ChatGoogleGenerativeAI

# TypedDict = plantilla del estado que LangGraph pasa de nodo en nodo.


class EstadoRouter(TypedDict):
    pedido: str       # texto original del usuario
    intencion: str    # etiqueta tras clasificar (la lee el router)
    respuesta: str    # texto que escribe la rama elegida


llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0.2,  # baja: clasificación más estable
)
print("Estado y modelo listos")
print(EstadoRouter.__annotations__)


Estado y modelo listos
{'pedido': <class 'str'>, 'intencion': <class 'str'>, 'respuesta': <class 'str'>}


## 5. Clasificar intención

El primer nodo **no responde al usuario**: solo etiqueta el pedido.

Cuatro intenciones alineadas con las tools del agente cultural (aquí solo son nombres; las tools reales vendrán después):

| Intención | Ejemplo de futura tool (más adelante) |
|-----------|---------------------------|
| `preferencias` | aclarar gustos / guía (`RAG_buscar_en_guia`) |
| `eventos` | qué hay en Madrid (`API_consultar_eventos_madrid`) |
| `horario` | cuándo / ahora (`hora_actual`) |
| `general` | saludo o pedido vago |

También usamos `texto_del_llm`: Gemini a veces devuelve `.content` como `str` y a veces como `list`; esta función lo unifica a texto.


In [3]:
def texto_del_llm(mensaje) -> str:
    """Gemini a veces devuelve .content como str y a veces como list."""
    content = mensaje.content
    if isinstance(content, list):
        partes = []
        for bloque in content:
            if isinstance(bloque, str):
                partes.append(bloque)
            elif isinstance(bloque, dict):
                partes.append(str(bloque.get("text", "")))
            else:
                partes.append(str(getattr(bloque, "text", bloque)))
        return "".join(partes).strip()
    return (content or "").strip()


def nodo_clasificar(estado: EstadoRouter) -> dict:
    """Paso 1: etiquetar el pedido (sin redactar la respuesta final)."""
    prompt = (
        "Clasifica el pedido de tarde cultural en UNA etiqueta:\n"
        "- preferencias (gustos, presupuesto, zona, tipo de plan)\n"
        "- eventos (qué hay, cartelera, actividades en Madrid)\n"
        "- horario (hora actual, cuándo salir, timing)\n"
        "- general (saludo o no está claro)\n\n"
        "Responde SOLO con una palabra: preferencias, eventos, horario o general.\n\n"
        f"Pedido: {estado['pedido']}"
    )
    raw = texto_del_llm(llm.invoke(prompt)).lower()

    # Si el modelo añade texto extra, buscamos la etiqueta en la respuesta
    etiqueta = "general"
    for cand in ("preferencias", "eventos", "horario", "general"):
        if cand in raw:
            etiqueta = cand
            break

    print(f"[clasificar] → {etiqueta} (raw={raw!r})")
    return {"intencion": etiqueta}  # solo actualizamos este campo


## 6. Nodos por rama (solo prompts distintos)

Cada rama es un nodo que **sí responde** al usuario. La diferencia entre ramas es el **system prompt**, no tools todavía.

`_responder` es un helper interno: llama al LLM y guarda el texto en `respuesta`. Los cuatro nodos solo cambian el mensaje de sistema.

En el futuro estos nodos podrían tener `tool_calls` y ejecutar tools reales.


In [4]:
def _responder(estado: EstadoRouter, sistema: str) -> dict:
    """Llama al LLM con un prompt de sistema y guarda la respuesta en el estado."""
    out = llm.invoke(f"{sistema}\n\nPedido: {estado['pedido']}")
    return {"respuesta": texto_del_llm(out)}


def nodo_preferencias(estado: EstadoRouter) -> dict:
    print("[rama] preferencias")
    return _responder(
        estado,
        "Ayuda a aclarar preferencias para una tarde cultural "
        "(intereses, presupuesto, zona). Máximo 4 frases. Sin inventar datos oficiales.",
    )


def nodo_eventos(estado: EstadoRouter) -> dict:
    print("[rama] eventos")
    return _responder(
        estado,
        "Habla como si fueras a consultar cartelera/eventos en Madrid. "
        "Explica qué mirarías (tipo de evento, zona). No inventes listados reales. Máximo 4 frases.",
    )


def nodo_horario(estado: EstadoRouter) -> dict:
    print("[rama] horario")
    return _responder(
        estado,
        "El usuario pregunta por timing/hora. Explica cómo usarías la hora actual "
        "para proponer una tarde. Máximo 3 frases. No des una hora inventada como si fuera real.",
    )


def nodo_general(estado: EstadoRouter) -> dict:
    print("[rama] general")
    return _responder(
        estado,
        "Saludo breve de planificador cultural. Pregunta 1 cosa útil (zona o intereses).",
    )


print("Ramas listas: preferencias, eventos, horario, general")


Ramas listas: preferencias, eventos, horario, general


## 7. Función de routing + grafo

Aquí unimos Nodos y Edges.

1. Registramos todos los nodos (`add_node`).
2. `START → clasificar` (edge fija).
3. `add_conditional_edges`: tras clasificar, la función `router` lee `estado["intencion"]` y elige el siguiente nodo.
4. Cada rama termina en `END`.
5. `compile()` deja el grafo listo para `invoke`.

In [5]:
from langgraph.graph import StateGraph, START, END


def router(estado: EstadoRouter) -> Literal["preferencias", "eventos", "horario", "general"]:
    """Devuelve el nombre del siguiente nodo según la intención ya clasificada."""
    return estado["intencion"]  # type: ignore[return-value]


grafo = StateGraph(EstadoRouter)

# --- nodos ---
grafo.add_node("clasificar", nodo_clasificar)
grafo.add_node("preferencias", nodo_preferencias)
grafo.add_node("eventos", nodo_eventos)
grafo.add_node("horario", nodo_horario)
grafo.add_node("general", nodo_general)

# --- edges ---
grafo.add_edge(START, "clasificar")

# Forma posicional (activa): source, path, path_map
grafo.add_conditional_edges(
    "clasificar",  # nodo de origen
    router,        # función que decide
    {              # valor del router → nodo destino
        "preferencias": "preferencias",
        "eventos": "eventos",
        "horario": "horario",
        "general": "general",
    },
)

# Misma llamada con nombres de parámetro (equivalente):
# grafo.add_conditional_edges(
#     source="clasificar",  # nodo de origen
#     path=router,          # función que decide
#     path_map={            # valor del router → nodo destino
#         "preferencias": "preferencias",
#         "eventos": "eventos",
#         "horario": "horario",
#         "general": "general",
#     },
# )

grafo.add_edge("preferencias", END)
grafo.add_edge("eventos", END)
grafo.add_edge("horario", END)
grafo.add_edge("general", END)

app = grafo.compile()
print("Grafo compilado")
print(app.get_graph().draw_ascii())


Grafo compilado
                                 +-----------+                                      
                                 | __start__ |                                      
                                 +-----------+                                      
                                        *                                           
                                        *                                           
                                        *                                           
                                 +------------+                                     
                               ..| clasificar |.                                    
                          .....  +------------+ ......                              
                   .......       ...        ..        ......                        
              .....             .             ..            ......                  
          ....                ..                .

## 8. Probar el agente de LangGraph

Primero definimos un helper `ejecutar_agente`: prepara el estado inicial, llama a `app.invoke` e imprime el resultado.

Luego ejecuta **cada celda de pedido por separado**. En cada una verás:

1. `[clasificar] → …` (la etiqueta)
2. `[rama] …` (el nodo elegido)
3. `intencion` y la `respuesta`

Solo pasamos el texto del `pedido`; el resto del estado lo rellenan los nodos.


In [6]:
def ejecutar_agente(pedido: str) -> dict:
    """Lanza el grafo con un pedido y muestra intención + respuesta."""
    final = app.invoke({"pedido": pedido, "intencion": "", "respuesta": ""})
    print("pedido:", pedido)
    print("intencion:", final["intencion"])
    print("respuesta:\n", final["respuesta"])
    return final


print("Helper listo: ejecutar_agente(pedido)")

Helper listo: ejecutar_agente(pedido)


In [7]:
# Pedido 1 — esperado: preferencias
ejecutar_agente("Me gustan museos baratos cerca de Sol")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[clasificar] → preferencias (raw='preferencias')
[rama] preferencias
pedido: Me gustan museos baratos cerca de Sol
intencion: preferencias
respuesta:
 Para organizar tu tarde, ¿prefieres enfocarte en arte clásico, historia o exposiciones contemporáneas? ¿Cuál es tu presupuesto máximo por entrada y qué día de la semana tienes pensado realizar la visita? Con esta información, podré sugerirte las opciones más cercanas a Sol que se ajusten a tus intereses.


{'pedido': 'Me gustan museos baratos cerca de Sol',
 'intencion': 'preferencias',
 'respuesta': 'Para organizar tu tarde, ¿prefieres enfocarte en arte clásico, historia o exposiciones contemporáneas? ¿Cuál es tu presupuesto máximo por entrada y qué día de la semana tienes pensado realizar la visita? Con esta información, podré sugerirte las opciones más cercanas a Sol que se ajusten a tus intereses.'}

In [8]:
# Pedido 2 — esperado: eventos
ejecutar_agente("¿Qué conciertos o exposiciones hay este finde?")

[clasificar] → eventos (raw='eventos')
[rama] eventos
pedido: ¿Qué conciertos o exposiciones hay este finde?
intencion: eventos
respuesta:
 Para saber qué hacer este fin de semana, primero filtraría la cartelera por tipo de evento, priorizando exposiciones en el eje del Paseo del Arte o conciertos en salas de Malasaña y Lavapiés. Revisaría las agendas culturales de medios locales para ver qué artistas independientes o muestras temporales coinciden con mis fechas. También echaría un ojo a las plataformas de venta de entradas para comprobar la disponibilidad en los espacios más céntricos. Con eso, decidiría si prefiero un plan cultural tranquilo o una noche de música en directo.


{'pedido': '¿Qué conciertos o exposiciones hay este finde?',
 'intencion': 'eventos',
 'respuesta': 'Para saber qué hacer este fin de semana, primero filtraría la cartelera por tipo de evento, priorizando exposiciones en el eje del Paseo del Arte o conciertos en salas de Malasaña y Lavapiés. Revisaría las agendas culturales de medios locales para ver qué artistas independientes o muestras temporales coinciden con mis fechas. También echaría un ojo a las plataformas de venta de entradas para comprobar la disponibilidad en los espacios más céntricos. Con eso, decidiría si prefiero un plan cultural tranquilo o una noche de música en directo.'}

In [9]:
# Pedido 3 — esperado: horario
ejecutar_agente("Si salgo a las 18h, ¿qué me da tiempo?")

[clasificar] → horario (raw='horario')
[rama] horario
pedido: Si salgo a las 18h, ¿qué me da tiempo?
intencion: horario
respuesta:
 Si sales a las 18:00, puedes aprovechar las horas de luz restantes para un paseo al aire libre o una actividad cultural que cierre tarde. Dependiendo de tu ubicación, ese margen es ideal para una cena temprana o asistir a un evento de tarde-noche. Todo dependerá de cuánto tiempo quieras dedicarle a cada plan antes de que termine la jornada.


{'pedido': 'Si salgo a las 18h, ¿qué me da tiempo?',
 'intencion': 'horario',
 'respuesta': 'Si sales a las 18:00, puedes aprovechar las horas de luz restantes para un paseo al aire libre o una actividad cultural que cierre tarde. Dependiendo de tu ubicación, ese margen es ideal para una cena temprana o asistir a un evento de tarde-noche. Todo dependerá de cuánto tiempo quieras dedicarle a cada plan antes de que termine la jornada.'}

In [10]:
# Pedido 4 — esperado: general
ejecutar_agente("Hola")

[clasificar] → general (raw='general')
[rama] general
pedido: Hola
intencion: general
respuesta:
 ¡Hola! Qué gusto saludarte. Soy tu planificador cultural y estoy aquí para ayudarte a organizar experiencias memorables.

Para empezar a diseñar tu agenda, ¿podrías decirme **en qué zona te encuentras o qué tipo de actividades te interesan más** (arte, historia, gastronomía, música, etc.)?


{'pedido': 'Hola',
 'intencion': 'general',
 'respuesta': '¡Hola! Qué gusto saludarte. Soy tu planificador cultural y estoy aquí para ayudarte a organizar experiencias memorables.\n\nPara empezar a diseñar tu agenda, ¿podrías decirme **en qué zona te encuentras o qué tipo de actividades te interesan más** (arte, historia, gastronomía, música, etc.)?'}

# 9. Variante sin `path_map`

Si el `router` ya devuelve el **nombre del nodo** (`"preferencias"`, `"eventos"`, …), puedes omitir el mapa.

Ejecuta esta celda: monta el mismo grafo en forma corta y lanza un pedido de prueba. El resultado debería ser el mismo tipo de flujo que con `path_map`.

`path_map` sí hace falta si la etiqueta y el nodo se llaman distinto (p. ej. `"math"` → `"matematicas"`).

In [11]:
# Mismo grafo, sin path_map (router ya devuelve el nombre del nodo)
grafo_corto = StateGraph(EstadoRouter)
grafo_corto.add_node("clasificar", nodo_clasificar)
grafo_corto.add_node("preferencias", nodo_preferencias)
grafo_corto.add_node("eventos", nodo_eventos)
grafo_corto.add_node("horario", nodo_horario)
grafo_corto.add_node("general", nodo_general)

grafo_corto.add_edge(START, "clasificar")
grafo_corto.add_conditional_edges("clasificar", router)  # sin path_map
grafo_corto.add_edge("preferencias", END)
grafo_corto.add_edge("eventos", END)
grafo_corto.add_edge("horario", END)
grafo_corto.add_edge("general", END)

app_corto = grafo_corto.compile()
print("Grafo sin path_map compilado")
print(app_corto.get_graph().draw_ascii())

# Prueba rápida (misma intención que en §8)
final = app_corto.invoke(
    {"pedido": "Me gustan museos baratos cerca de Sol", "intencion": "", "respuesta": ""}
)
print("\nintencion:", final["intencion"])
print("respuesta:\n", final["respuesta"])

Grafo sin path_map compilado
                                 +-----------+                                      
                                 | __start__ |                                      
                                 +-----------+                                      
                                        *                                           
                                        *                                           
                                        *                                           
                                 +------------+                                     
                               ..| clasificar |.                                    
                          .....  +------------+ ......                              
                   .......       ...        ..        ......                        
              .....             .             ..            ......                  
          ....                ..    

## 9. Para llevarnos

| Pieza | Rol |
|-------|-----|
| `TypedDict` | Contrato del estado (`pedido`, `intencion`, `respuesta`) |
| `nodo_clasificar` | Escribe la etiqueta en el estado |
| `router` | Devuelve el **nombre** del siguiente nodo |
| `add_conditional_edges` | Bifurca según lo que diga `router` |
| Ramas | Mismo patrón que “¿hay tool calls?” en un agente con tools |

En siguientes pasos terminaremos el agente cultural completo con LangGraph. Veremos que el grafo típico será `agent ↔ tools` con una condición sobre function calling. 
